# Merging the Datasets

I am using the Kaggle MBTI dataset and the DailyDialog dataset to fine-tune my LLM. Earlier I labelled the DailyDialog dataset with MBTI types, so now I need to merge them into one dataset to make fine-tuning easy.

I'm taking the datasets from Huggingface, except for the dataset containing the prepared turns of the dialogs. These were prepared in the dialogue_data_exploration.ipynb notebook.

In [ ]:
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict, ClassLabel, concatenate_datasets
from sklearn.model_selection import train_test_split
import os
import pandas as pd 
import re
from dotenv import load_dotenv

rseed = 42

In [ ]:
load_dotenv()
login(os.getenv("HF-TOKEN"))


In [ ]:
#loading datasets
mbti_dict = load_dataset("DrinkIcedT/MBTI_balanced_pub_new")
mbti_train = mbti_dict["train"]
mbti_val = mbti_dict["validation"]
mbti_test = mbti_dict["test"]


dd = load_dataset("DrinkIcedT/dailydialog_mbti_labeled")
dd = dd["train"]
df = dd.to_pandas()


In [ ]:
# pivot table, so instead of having "person | label" we get "person A | person B | label person A | label Person B"
dd_piv = df.pivot(index="idx", columns= ["person"], values = ["post", "label"])
dd_piv.columns = [f"{val} {col}" for val, col in dd_piv.columns]
dd_piv = dd_piv.rename(columns={"post A": "person A", "post B": "person B"})
dd_piv.columns

print(dd_piv.head())

In [ ]:
# making a list out of the dialog turns
turns = pd.read_csv("../data/csv/dailydialog_turns.csv", sep=",")
turns = turns.drop(columns=["Person A", "Person B"])

print(turns["turns"][0])

# take everything thats between '' or ""
pattern = r"""(?<!\w)['"](.*?)(?<!\w)['"]"""

turns["turns"] = turns["turns"].apply(
    lambda x: re.findall(pattern, x) if isinstance(x, str) else x
)
print(turns["turns"][0])

In [ ]:
# merge label cols with turns
dd_final = turns.merge(dd_piv[["label A", "label B"]], left_index=True, right_index=True)

In [ ]:
def prepare_samples(dialog_turns, mbti_a, mbti_b, context_window=3):
    """
    dialog_turns: list of strings, A and B taking turns, A starts
    mbti_a, mbti_b: MBTI labels
    context_window: how many turns as dialogue context
    """
    samples = []
    
    for i in range(1, len(dialog_turns), 2):  # all B-turns (odd indices)
        # context: last context_window Turns before current B turn
        start = max(0, i - context_window)
        context_turns = dialog_turns[start:i]
        
        # context to string
        #speakers = ["A", "B"] * len(context_turns)  # A starts
        context_str = "\n".join(
            f"{'A' if j % 2 == 0 else 'B'}: {turn}"
            for j, turn in enumerate(context_turns, start=start)
        )
        
        samples.append({
            "system": f"You are a person (Person B) with the personality type {mbti_b}. "
                      f"You are talking to a person (Person A) with the personality type {mbti_a}."
                      f"You see here the conversation history."
                      f"Answer as Person B in the next turn as a person with personality type {mbti_b} might.",
            "user": context_str,
            "assistant": dialog_turns[i]
        })
    
    return samples

In [ ]:
def clean_utterance(text):
    text = text.strip()
    # delete white spaces before punctuation marks
    text = re.sub(r'\s([?.!,;:\'])', r'\1', text)
    text = re.sub(r"\s*’\s*", "'", text)
    # include whitespace after punctuation marks
    text = re.sub(r'([?.!])([A-Z])', r'\1 \2', text)
    return text

dd_final["turns"] = dd_final["turns"].apply(lambda turns: [clean_utterance(t) for t in turns])

In [ ]:
print(dd_final["turns"][0])
print(len(dd_final["turns"][0]))
print(dd_final.columns.tolist())

In [ ]:
all_samples = []

for _, row in dd_final.iterrows():
    turns = row["turns"]  # list of turns
    samples = prepare_samples(
        dialog_turns=turns,
        mbti_a=row["label A"],
        mbti_b=row["label B"],
        context_window=3
    )
    all_samples.extend(samples)
    
    # flipped roles --> more data
    samples_flipped = prepare_samples(
        dialog_turns=turns,
        mbti_a=row["label B"],
        mbti_b=row["label A"],
        context_window=3
    )
    all_samples.extend(samples_flipped)

result_df = pd.DataFrame(all_samples)
print(f"Total samples: {len(result_df)}")

In [ ]:
# filter mbti types from the system message
result_df["labels"] = result_df["system"].str.extract(r"You are a person \(Person B\) with the personality type (\w+)")

# huggingface dict
df_hf = Dataset.from_pandas(result_df, preserve_index=False)

# encode columns
df_hf = df_hf.class_encode_column("labels")
mbti_labels = ["ENFJ", "ENFP", "ENTJ", "ENTP", "ESFJ", "ESFP", "ESTJ", "ESTP", 
               "INFJ", "INFP", "INTJ", "INTP", "ISFJ", "ISFP", "ISTJ", "ISTP"]

new_features = df_hf.features.copy()
new_features["labels"] = ClassLabel(names=mbti_labels)

df_hf = df_hf.cast(new_features)

# split
split = df_hf.train_test_split(test_size = 0.2, shuffle = True, stratify_by_column="labels", seed = rseed)
train = split["train"]
df_temp = split["test"]

split2 = df_temp.train_test_split(test_size = 0.5, shuffle = True, stratify_by_column= "labels", seed = rseed)
val = split2["train"]
test = split2["test"]

dd_dict = DatasetDict({
    "train": train,
    "test": test,
    "validation": val
})




In [ ]:
# convert data according to chat template of the model
# Kaggle
def convert_mbti_to_chatml(example):
    mbti_type = mbti_dict["train"].features["labels"].int2str(example["labels"])
    prompt = f"Your personality type is {mbti_type}. Share your thoughts on a topic that's been on your mind recently."
    return {
        "messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": example["post"]}
        ]
    }

# DailyDialog
def convert_dd_to_chatml(example):
    mbti_type = dd_dict["train"].features["labels"].int2str(example["labels"])
    prompt = (f"Your personality type is {mbti_type}. "
              f"You are in a conversation. Here is the conversation history:\n"
              f"{example['user']}\n"
              f"Respond in the next turn.")
    return {
        "messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": example["assistant"]}
        ]
    }

# convert
mbti_chatml = mbti_dict.map(convert_mbti_to_chatml)
dd_chatml = dd_dict.map(convert_dd_to_chatml)

# drop everything except for the messages
columns_to_keep = ["messages"]

mbti_chatml = mbti_chatml.remove_columns(
    [c for c in mbti_chatml["train"].column_names if c not in columns_to_keep]
)
dd_chatml = dd_chatml.remove_columns(
    [c for c in dd_chatml["train"].column_names if c not in columns_to_keep]
)

# Concatenate and shuffle
from datasets import concatenate_datasets, DatasetDict

combined = DatasetDict({
    "train": concatenate_datasets([mbti_chatml["train"], dd_chatml["train"]]).shuffle(seed=rseed),
    "validation": concatenate_datasets([mbti_chatml["validation"], dd_chatml["validation"]]).shuffle(seed=rseed),
    "test": concatenate_datasets([mbti_chatml["test"], dd_chatml["test"]]).shuffle(seed=rseed)
})

In [ ]:
train_c = combined["train"]

In [ ]:
print(f"Available splits: {list(combined.keys())}")


def has_valid_assistant_content(example, min_length=3):
    """
    Tests for valid assistant messages
    """
    for m in example["messages"]:
        if m.get("role") == "assistant":
            content = m.get("content", "")
            if not content or len(content.strip()) < min_length:
                return False
    return True


print("\nFilter empty/very short answer from assitant column...")
for split_name in list(combined.keys()):
    original_size = len(combined[split_name])
    combined[split_name] = combined[split_name].filter(has_valid_assistant_content)
    new_size = len(combined[split_name])
    removed = original_size - new_size
    pct = (removed / original_size * 100) if original_size > 0 else 0
    print(f"  {split_name}: {original_size} -> {new_size} (entfernt: {removed}, {pct:.3f}%)")

print("\nDone!")

In [ ]:
combined.push_to_hub("DrinkIcedT/mbti_dialogue_pub_filtered")